# 2 Assets

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

In [2]:
stocks = ["AAPL", "TSLA"]

data = yf.download(
    stocks, 
    start = "2015-01-01",
    end = "2026-01-01",
    interval = "1d",
    auto_adjust = True # requests prices adjusted for corporote actions like splits and dividends
)

print(data.head())
print(data.shape)

[*********************100%***********************]  2 of 2 completed

Price           Close                  High                   Low             \
Ticker           AAPL       TSLA       AAPL       TSLA       AAPL       TSLA   
Date                                                                           
2015-01-02  24.171757  14.620667  24.638257  14.883333  23.733998  14.217333   
2015-01-05  23.490803  14.006000  24.021419  14.433333  23.305088  13.810667   
2015-01-06  23.493008  14.085333  23.751682  14.280000  23.132630  13.614000   
2015-01-07  23.822441  14.063333  23.921931  14.318667  23.590296  13.985333   
2015-01-08  24.737747  14.041333  24.795231  14.253333  24.032470  14.000667   

Price            Open                Volume            
Ticker           AAPL       TSLA       AAPL      TSLA  
Date                                                   
2015-01-02  24.627201  14.858000  212818400  71466000  
2015-01-05  23.941826  14.303333  257142000  80527500  
2015-01-06  23.554913  14.004000  263188400  93928500  
2015-01-07  23.700841  

In [3]:
prices = data["Close"]

In [4]:
print(prices.head()) #displays firsy 5 rows
print(prices.shape) # returns number of rows and columns
print(prices.isna().sum()) # identifies and counts missing values

Ticker           AAPL       TSLA
Date                            
2015-01-02  24.171757  14.620667
2015-01-05  23.490803  14.006000
2015-01-06  23.493008  14.085333
2015-01-07  23.822441  14.063333
2015-01-08  24.737747  14.041333
(2766, 2)
Ticker
AAPL    0
TSLA    0
dtype: int64


## Logarithmic Daily Returns

In [5]:
returns = np.log(prices / prices.shift(1)) #.shift shifts prices down one day so each days price can be compared with previous days price
returns = returns.dropna()
print(returns.head())
print(returns.shape)

Ticker          AAPL      TSLA
Date                          
2015-01-05 -0.028576 -0.042950
2015-01-06  0.000094  0.005648
2015-01-07  0.013925 -0.001563
2015-01-08  0.037702 -0.001566
2015-01-09  0.001072 -0.018981
(2765, 2)


The 1st observation has no preceeding trading-day price, so its logarithmic return cannot be calculated, why the closing price observation has one extra value. 

In [6]:
print(returns.describe())

Ticker         AAPL         TSLA
count   2765.000000  2765.000000
mean       0.000874     0.001239
std        0.018154     0.036274
min       -0.137708    -0.236518
25%       -0.007309    -0.016677
50%        0.000950     0.001257
75%        0.009964     0.019394
max        0.142617     0.204491


## Sample Covariance Matrix

In [7]:
mean_returns = returns.mean() # calculate mean of each stock
print(mean_returns)
centered_returns = returns - mean_returns
n = len(returns) # no observations
cov_manual = centered_returns.T @ centered_returns / (n-1)

print(cov_manual)

Ticker
AAPL    0.000874
TSLA    0.001239
dtype: float64
Ticker      AAPL      TSLA
Ticker                    
AAPL    0.000330  0.000286
TSLA    0.000286  0.001316


In [8]:
cov_pandas = returns.cov()
print(cov_pandas)

Ticker      AAPL      TSLA
Ticker                    
AAPL    0.000330  0.000286
TSLA    0.000286  0.001316


In [9]:
print(np.allclose(cov_manual, cov_pandas))

True


## Sample Correlation Matrix

In [10]:
std_returns = returns.std() # sample standard deviation
corr_manual = cov_manual / np.outer(std_returns, std_returns)
print(corr_manual)

Ticker      AAPL      TSLA
Ticker                    
AAPL    1.000000  0.435021
TSLA    0.435021  1.000000


In [11]:
corr_pandas = returns.corr()
print(corr_pandas)

Ticker      AAPL      TSLA
Ticker                    
AAPL    1.000000  0.435021
TSLA    0.435021  1.000000


In [12]:
print(np.allclose(corr_manual, corr_pandas))

True


## Verify Mathematical Properties

In [13]:
print(np.allclose(cov_pandas, cov_pandas.T))
print(np.allclose(corr_pandas, corr_pandas.T))

True
True


They are symmetric

In [14]:
print(np.linalg.eigvals(cov_pandas))
print(np.linalg.eigvals(corr_pandas))

[0.00025241 0.00139301]
[1.43502084 0.56497916]


Valid, the eigenvalues are non-negative and within numerical tolerance

In [15]:
print(sum(np.linalg.eigvals(corr_pandas)))

2.0


Sum of the eigenvalues of the correlation matrix equals the number of assets

In [16]:
eigenvectors = np.linalg.eigh(corr_pandas)[1]
print(eigenvectors)

[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


In [17]:
print(np.linalg.eigvalsh(cov_pandas).min() >= -1e-10)
print(np.linalg.eigvalsh(corr_pandas).min() >= -1e-10)

True
True


In [18]:
print((corr_pandas >= -1).all().all())
print((corr_pandas <= 1).all().all())

True
True


Pearson coeffiecent lies between -1 and 1

In [19]:
returns.to_csv("returns_AAPL_TSLA.csv")

## Visuals

The heatmap provides a visual representation of the Pearson correlation matrix.

In [20]:
fig = px.imshow(
    corr_pandas,
    text_auto=".3f",
    color_continuous_scale="piyg_r",
    zmin=-1,
    zmax=1,
    title="Apple and Tesla Correlation Matrix"
)

fig.show()

This scatterplot visualises the relationship between Apple's and Tesla's daily logarithmic returns.
- X-axis: Apple's daily log returns.
- Y-axis: Tesla's daily log returns.
- Pink points: Individual trading-day observations.
- Purple line: The line of best fit.
  
Each point represents the returns of both stocks on the same trading day.

In [21]:
r = corr_pandas.loc["AAPL", "TSLA"]

fig = px.scatter(
    returns,
    x="AAPL",
    y="TSLA",
    trendline="ols",
    color_discrete_sequence=["hotpink"],
    trendline_color_override="purple",
    title=f"Apple vs Tesla Daily Returns (r = {r:.3f})"
)

fig.show()

This graph shows how Apple's and Tesla's daily logarithmic returns fluctuate throughout the dataset.
- X-axis: Trading days (or dates if your DataFrame has a date index).
- Y-axis: Daily logarithmic returns.
- Pink line: Apple.
- Blue line: Tesla.

In [22]:
fig = px.line(
    returns,
    y=["AAPL", "TSLA"],
    title="Apple and Tesla Daily Log Returns (2015–2025)",
    color_discrete_map={
        "AAPL": "hotpink",
        "TSLA": "royalblue"
    }
)

fig.update_layout(
    xaxis_title="Trading Day",
    yaxis_title="Daily Log Return",
    legend_title="Stock"
)

fig.show()

In [23]:
cov_pandas.to_csv("python_cov_2.csv")
corr_pandas.to_csv("python_corr_2.csv")

# 5 Assets

In [24]:
stocks = ["AAPL", "TSLA", "MSFT", "KO", "JNJ" ]

data = yf.download(
    stocks, 
    start = "2015-01-01",
    end = "2026-01-01",
    interval = "1d",
    auto_adjust = True # requests prices adjusted for corporote actions like splits and dividends
)

[*********************100%***********************]  5 of 5 completed


In [25]:
print(data.head())
print(data.shape)

Price           Close                                                   High  \
Ticker           AAPL        JNJ         KO       MSFT       TSLA       AAPL   
Date                                                                           
2015-01-02  24.171757  75.737373  29.215925  39.607182  14.620667  24.638257   
2015-01-05  23.490803  75.208405  29.215925  39.242954  14.006000  24.021419   
2015-01-06  23.493008  74.838860  29.437786  38.666965  14.085333  23.751682   
2015-01-07  23.822441  76.490982  29.805241  39.158253  14.063333  23.921931   
2015-01-08  24.737747  77.092392  30.165764  40.310226  14.041333  24.795231   

Price                                                   ...       Open  \
Ticker            JNJ         KO       MSFT       TSLA  ...       AAPL   
Date                                                    ...              
2015-01-02  76.483738  29.396186  40.166222  14.883333  ...  24.627201   
2015-01-05  75.889550  29.791371  39.581765  14.433333  ...  23

In [26]:
prices = data["Close"]

## Logarithmic Daily Returns

In [27]:
returns = np.log(prices / prices.shift(1)) #.shift shifts prices down one day so each days price can be compared with previous days price
returns = returns.dropna()
print(returns.head())
print(returns.shape)

Ticker          AAPL       JNJ        KO      MSFT      TSLA
Date                                                        
2015-01-05 -0.028576 -0.007009  0.000000 -0.009239 -0.042950
2015-01-06  0.000094 -0.004926  0.007565 -0.014786  0.005648
2015-01-07  0.013925  0.021836  0.012405  0.012626 -0.001563
2015-01-08  0.037702  0.007832  0.012023  0.028994 -0.001566
2015-01-09  0.001072 -0.013722 -0.011093 -0.008441 -0.018981
(2765, 5)


In [28]:
print(returns.describe())

Ticker         AAPL          JNJ           KO         MSFT         TSLA
count   2765.000000  2765.000000  2765.000000  2765.000000  2765.000000
mean       0.000874     0.000358     0.000309     0.000903     0.001239
std        0.018154     0.011481     0.011230     0.016921     0.036274
min       -0.137708    -0.105781    -0.101728    -0.159454    -0.236518
25%       -0.007309    -0.004884    -0.004719    -0.006750    -0.016677
50%        0.000950     0.000462     0.000481     0.000954     0.001257
75%        0.009964     0.005981     0.005869     0.009384     0.019394
max        0.142617     0.076940     0.062783     0.132929     0.204491


## Sample Covariance Matrix

In [29]:
mean_returns = returns.mean() # calculate mean of each stock
print(mean_returns)
centered_returns = returns - mean_returns
n = len(returns) # no observations
cov_manual = centered_returns.T @ centered_returns / (n-1)

print(cov_manual)

Ticker
AAPL    0.000874
JNJ     0.000358
KO      0.000309
MSFT    0.000903
TSLA    0.001239
dtype: float64
Ticker      AAPL       JNJ        KO      MSFT      TSLA
Ticker                                                  
AAPL    0.000330  0.000066  0.000070  0.000204  0.000286
JNJ     0.000066  0.000132  0.000065  0.000065  0.000040
KO      0.000070  0.000065  0.000126  0.000069  0.000057
MSFT    0.000204  0.000065  0.000069  0.000286  0.000252
TSLA    0.000286  0.000040  0.000057  0.000252  0.001316


In [30]:
cov_pandas = returns.cov()
print(cov_pandas)

Ticker      AAPL       JNJ        KO      MSFT      TSLA
Ticker                                                  
AAPL    0.000330  0.000066  0.000070  0.000204  0.000286
JNJ     0.000066  0.000132  0.000065  0.000065  0.000040
KO      0.000070  0.000065  0.000126  0.000069  0.000057
MSFT    0.000204  0.000065  0.000069  0.000286  0.000252
TSLA    0.000286  0.000040  0.000057  0.000252  0.001316


In [31]:
print(np.allclose(cov_manual, cov_pandas))

True


## Sample Correlation Matrix

In [32]:
std_returns = returns.std() # sample standard deviation
corr_manual = cov_manual / np.outer(std_returns, std_returns)
print(corr_manual)

Ticker      AAPL       JNJ        KO      MSFT      TSLA
Ticker                                                  
AAPL    1.000000  0.316591  0.344006  0.664803  0.435021
JNJ     0.316591  1.000000  0.502654  0.333894  0.095266
KO      0.344006  0.502654  1.000000  0.362113  0.139512
MSFT    0.664803  0.333894  0.362113  1.000000  0.410569
TSLA    0.435021  0.095266  0.139512  0.410569  1.000000


In [33]:
corr_pandas = returns.corr()
print(corr_pandas)

Ticker      AAPL       JNJ        KO      MSFT      TSLA
Ticker                                                  
AAPL    1.000000  0.316591  0.344006  0.664803  0.435021
JNJ     0.316591  1.000000  0.502654  0.333894  0.095266
KO      0.344006  0.502654  1.000000  0.362113  0.139512
MSFT    0.664803  0.333894  0.362113  1.000000  0.410569
TSLA    0.435021  0.095266  0.139512  0.410569  1.000000


In [34]:
print(np.allclose(corr_manual, corr_pandas))

True


## Verify Mathematical Properties

In [35]:
print(np.allclose(cov_pandas, cov_pandas.T))
print(np.allclose(corr_pandas, corr_pandas.T))

True
True


They are Symmetric

In [36]:
print(np.linalg.eigvals(cov_pandas))
print(np.linalg.eigvals(corr_pandas))

[1.47845601e-03 4.05512136e-04 1.39963843e-04 1.01863806e-04
 6.38844836e-05]
[2.48369697 1.10408368 0.33425768 0.58271864 0.49524303]


Non-negative eigenvalues

In [37]:
print(sum(np.linalg.eigvals(corr_pandas)))

4.999999999999996


In [38]:
eigenvectors = np.linalg.eigh(corr_pandas)[1]
print(eigenvectors)

[[ 0.71313448  0.00452174 -0.40512616  0.23858814 -0.51996852]
 [ 0.00486352 -0.70353182  0.16993416 -0.56765202 -0.39231733]
 [ 0.01653443  0.70545521  0.27151522 -0.50610711 -0.41496342]
 [-0.69975113  0.02097581 -0.445622    0.19572625 -0.52251377]
 [-0.03860747 -0.08315943  0.73122889  0.57130877 -0.36125417]]


In [39]:
print(np.linalg.eigvalsh(cov_pandas).min() >= -1e-10)
print(np.linalg.eigvalsh(corr_pandas).min() >= -1e-10)

True
True


In [40]:
eigenvectors = np.linalg.eigh(corr_pandas)[1]

eigenvectors_df = pd.DataFrame(
    eigenvectors,
    index=corr_pandas.index,
    columns=["Eigenvector 1", "Eigenvector 2",
             "Eigenvector 3", "Eigenvector 4", "Eigenvector 5"]
)

print(eigenvectors_df)

        Eigenvector 1  Eigenvector 2  Eigenvector 3  Eigenvector 4  \
Ticker                                                               
AAPL         0.713134       0.004522      -0.405126       0.238588   
JNJ          0.004864      -0.703532       0.169934      -0.567652   
KO           0.016534       0.705455       0.271515      -0.506107   
MSFT        -0.699751       0.020976      -0.445622       0.195726   
TSLA        -0.038607      -0.083159       0.731229       0.571309   

        Eigenvector 5  
Ticker                 
AAPL        -0.519969  
JNJ         -0.392317  
KO          -0.414963  
MSFT        -0.522514  
TSLA        -0.361254  


In [41]:
print((corr_pandas >= -1).all().all())
print((corr_pandas <= 1).all().all())

True
True


Pearson correlation coefficent lies between -1 and +1

## Visuals

In [42]:
fig = px.imshow(
    corr_pandas,
    text_auto=".3f",
    color_continuous_scale="piyg_r",
    zmin=-1,
    zmax=1,
    title="5 Asset Correlation Matrix"
)

fig.show()

In [43]:
fig = px.scatter_matrix(
    returns,
    dimensions=["AAPL", "TSLA", "MSFT", "KO", "JNJ"],
    color_discrete_sequence=["hotpink", "purple", "orchid", "deeppink", "mediumpurple"],
    title="Correlation Between Five Stocks"
)

fig.show()

In [44]:
from itertools import combinations

stocks = ["AAPL", "TSLA", "MSFT", "KO", "JNJ"]
colours = ["hotpink", "purple", "orchid", "deeppink", "mediumpurple"]

for i, (a, b) in enumerate(combinations(stocks, 2)):
    r = corr_pandas.loc[a, b]

    fig = px.scatter(
        returns,
        x=a,
        y=b,
        trendline="ols",
        color_discrete_sequence=[colours[i % len(colours)]],
        trendline_color_override="purple",
        title=f"{a} vs {b} (r = {r:.3f})"
    )

    fig.show()

In [45]:
fig = px.line(
    returns,
    y=["AAPL", "TSLA", "MSFT", "KO", "JNJ"],
    title="Daily Log Returns for Apple, Tesla, Microsoft, Coca-Cola and Johnson & Johnson (2015–2025)",
    color_discrete_map={
        "AAPL": "hotpink",
        "TSLA": "royalblue",
        "MSFT": "indigo",
        "KO": "red",
        "JNJ": "teal"
    }
)

fig.update_layout(
    xaxis_title="Trading Day",
    yaxis_title="Daily Log Return",
    legend_title="Stock"
)

fig.show()

In [46]:
import plotly.express as px
from ipywidgets import interact

stocks = ["AAPL", "TSLA", "MSFT", "KO", "JNJ"]

@interact(x=stocks, y=stocks, z=stocks)
def scatter_3d(x="AAPL", y="TSLA", z="MSFT"):
    fig = px.scatter_3d(
        returns,
        x=x,
        y=y,
        z=z,
        color_discrete_sequence=["hotpink"],
        title=f"3D Scatter Plot: {x} vs {y} vs {z}"
    )
    fig.update_traces(marker=dict(size=3, opacity=0.5))
    fig.show()

interactive(children=(Dropdown(description='x', options=('AAPL', 'TSLA', 'MSFT', 'KO', 'JNJ'), value='AAPL'), …

Each point in this plot represents one trading day's returns for all 3 stocks that are selected in the dropdown menus. 

What the distribution means:

1. Positive Correlations: example: Apple, Microsoft, Tesla.

   Points form an elongated cloud extentding diagonally through 3D space. The stringer the linear relationship, the more structured the cloud may         appear.

3. Weak Correlations: example: Apple, Coca-Cola, Johnson & Johnson

   If stocks have relatively weak correlations, the points may form a more dispersed cloud without a clear diagonal pattern. This suggests that their      daily returns do not consistently move together in a linear relationship.
   Stocks with lower correlations can offer more diversification benefits because their returns are less closely related. Low correlation alone doesn't    guarentee a low-risk portfolio.

5. Volatility Differences

   If one stock is more volatile than another, its returns would spread accross a much wider range on its axis in comparison to the other stock. 

   


In [47]:
@interact(stock1=stocks, stock2=stocks, stock3=stocks)
def correlation_heatmap(stock1="AAPL", stock2="TSLA", stock3="MSFT"):

    selected = [stock1, stock2, stock3]

    corr = returns[selected].corr()

    fig = px.imshow(
        corr,
        text_auto=".3f",
        color_continuous_scale="RdPu",
        zmin=-1,
        zmax=1,
        title=f"Correlation Matrix: {stock1}, {stock2}, {stock3}"
    )

    fig.show()

interactive(children=(Dropdown(description='stock1', options=('AAPL', 'TSLA', 'MSFT', 'KO', 'JNJ'), value='AAP…

In [48]:
cov_pandas.to_csv("python_cov_5.csv")
corr_pandas.to_csv("python_corr_5.csv")

# 10 Assets

## Logarithmic Daily Returns

In [49]:
stocks = ["AAPL", "TSLA", "MSFT", "KO", "JNJ", "NFLX", "V", "BA", "DIS", "CVX"]

data = yf.download(
    stocks, 
    start = "2015-01-01",
    end = "2026-01-01",
    interval = "1d",
    auto_adjust = True # requests prices adjusted for corporote actions like splits and dividends
)

[*********************100%***********************]  10 of 10 completed


In [50]:
print(data.head())
print(data.shape)

Price           Close                                                          \
Ticker           AAPL          BA        CVX        DIS        JNJ         KO   
Date                                                                            
2015-01-02  24.171757  113.657219  68.302231  84.716797  75.737373  29.215925   
2015-01-05  23.490803  112.870056  65.572105  83.478806  75.208405  29.215925   
2015-01-06  23.493008  111.540627  65.541763  83.036018  74.838860  29.437786   
2015-01-07  23.822441  113.272392  65.487160  83.885452  76.490982  29.805241   
2015-01-08  24.737747  115.275261  66.985703  84.752953  77.092392  30.165764   

Price                                                  ...     Volume  \
Ticker           MSFT      NFLX       TSLA          V  ...       AAPL   
Date                                                   ...              
2015-01-02  39.607182  4.984857  14.620667  60.974831  ...  212818400   
2015-01-05  39.242954  4.731143  14.006000  59.628872  ... 

In [51]:
prices = data["Close"]

In [52]:
print(prices.head())

Ticker           AAPL          BA        CVX        DIS        JNJ         KO  \
Date                                                                            
2015-01-02  24.171757  113.657219  68.302231  84.716797  75.737373  29.215925   
2015-01-05  23.490803  112.870056  65.572105  83.478806  75.208405  29.215925   
2015-01-06  23.493008  111.540627  65.541763  83.036018  74.838860  29.437786   
2015-01-07  23.822441  113.272392  65.487160  83.885452  76.490982  29.805241   
2015-01-08  24.737747  115.275261  66.985703  84.752953  77.092392  30.165764   

Ticker           MSFT      NFLX       TSLA          V  
Date                                                   
2015-01-02  39.607182  4.984857  14.620667  60.974831  
2015-01-05  39.242954  4.731143  14.006000  59.628872  
2015-01-06  38.666965  4.650143  14.085333  59.244663  
2015-01-07  39.158253  4.674286  14.063333  60.038429  
2015-01-08  40.310226  4.778000  14.041333  60.843689  


In [53]:
print(prices.shape)

(2766, 10)


In [54]:
print(prices.isna().sum())

Ticker
AAPL    0
BA      0
CVX     0
DIS     0
JNJ     0
KO      0
MSFT    0
NFLX    0
TSLA    0
V       0
dtype: int64


In [55]:
returns = np.log(prices / prices.shift(1))
returns = returns.dropna()

In [56]:
print(returns.head())
print(returns.shape)

Ticker          AAPL        BA       CVX       DIS       JNJ        KO  \
Date                                                                     
2015-01-05 -0.028576 -0.006950 -0.040792 -0.014721 -0.007009  0.000000   
2015-01-06  0.000094 -0.011848 -0.000463 -0.005318 -0.004926  0.007565   
2015-01-07  0.013925  0.015407 -0.000833  0.010178  0.021836  0.012405   
2015-01-08  0.037702  0.017527  0.022625  0.010288  0.007832  0.012023   
2015-01-09  0.001072 -0.001974 -0.020127  0.004893 -0.013722 -0.011093   

Ticker          MSFT      NFLX      TSLA         V  
Date                                                
2015-01-05 -0.009239 -0.052238 -0.042950 -0.022321  
2015-01-06 -0.014786 -0.017269  0.005648 -0.006464  
2015-01-07  0.012626  0.005178 -0.001563  0.013309  
2015-01-08  0.028994  0.021946 -0.001566  0.013323  
2015-01-09 -0.008441 -0.015578 -0.018981 -0.014934  
(2765, 10)


In [57]:
print(returns.describe())

Ticker         AAPL           BA          CVX          DIS          JNJ  \
count   2765.000000  2765.000000  2765.000000  2765.000000  2765.000000   
mean       0.000874     0.000234     0.000280     0.000104     0.000358   
std        0.018154     0.025279     0.018468     0.017623     0.011481   
min       -0.137708    -0.272444    -0.250062    -0.141139    -0.105781   
25%       -0.007309    -0.009944    -0.007811    -0.007239    -0.004884   
50%        0.000950     0.000409     0.000660     0.000089     0.000462   
75%        0.009964     0.010954     0.008746     0.008129     0.005981   
max        0.142617     0.217677     0.204904     0.134639     0.076940   

Ticker           KO         MSFT         NFLX         TSLA            V  
count   2765.000000  2765.000000  2765.000000  2765.000000  2765.000000  
mean       0.000309     0.000903     0.001061     0.001239     0.000631  
std        0.011230     0.016921     0.027295     0.036274     0.015271  
min       -0.101728    -0.15

## Sample Covariance Matrix

In [58]:
mean_returns = returns.mean() # calculate mean of each stock
print(mean_returns)
centered_returns = returns - mean_returns
n = len(returns) # no observations
cov_manual = centered_returns.T @ centered_returns / (n-1)

print(cov_manual)

Ticker
AAPL    0.000874
BA      0.000234
CVX     0.000280
DIS     0.000104
JNJ     0.000358
KO      0.000309
MSFT    0.000903
NFLX    0.001061
TSLA    0.001239
V       0.000631
dtype: float64
Ticker      AAPL        BA       CVX       DIS       JNJ        KO      MSFT  \
Ticker                                                                         
AAPL    0.000330  0.000187  0.000114  0.000136  0.000066  0.000070  0.000204   
BA      0.000187  0.000639  0.000224  0.000209  0.000074  0.000105  0.000165   
CVX     0.000114  0.000224  0.000341  0.000138  0.000071  0.000080  0.000108   
DIS     0.000136  0.000209  0.000138  0.000311  0.000056  0.000073  0.000132   
JNJ     0.000066  0.000074  0.000071  0.000056  0.000132  0.000065  0.000065   
KO      0.000070  0.000105  0.000080  0.000073  0.000065  0.000126  0.000069   
MSFT    0.000204  0.000165  0.000108  0.000132  0.000065  0.000069  0.000286   
NFLX    0.000203  0.000172  0.000079  0.000152  0.000042  0.000043  0.000211   
TSLA    

In [59]:
cov_pandas = returns.cov()
print(cov_pandas)

Ticker      AAPL        BA       CVX       DIS       JNJ        KO      MSFT  \
Ticker                                                                         
AAPL    0.000330  0.000187  0.000114  0.000136  0.000066  0.000070  0.000204   
BA      0.000187  0.000639  0.000224  0.000209  0.000074  0.000105  0.000165   
CVX     0.000114  0.000224  0.000341  0.000138  0.000071  0.000080  0.000108   
DIS     0.000136  0.000209  0.000138  0.000311  0.000056  0.000073  0.000132   
JNJ     0.000066  0.000074  0.000071  0.000056  0.000132  0.000065  0.000065   
KO      0.000070  0.000105  0.000080  0.000073  0.000065  0.000126  0.000069   
MSFT    0.000204  0.000165  0.000108  0.000132  0.000065  0.000069  0.000286   
NFLX    0.000203  0.000172  0.000079  0.000152  0.000042  0.000043  0.000211   
TSLA    0.000286  0.000290  0.000149  0.000194  0.000040  0.000057  0.000252   
V       0.000155  0.000182  0.000130  0.000143  0.000072  0.000081  0.000158   

Ticker      NFLX      TSLA         V  


In [60]:
print(np.allclose(cov_manual, cov_pandas))

True


## Sample Correlation Matrix

In [61]:
std_returns = returns.std() # sample standard deviation
corr_manual = cov_manual / np.outer(std_returns, std_returns)
print(corr_manual)

Ticker      AAPL        BA       CVX       DIS       JNJ        KO      MSFT  \
Ticker                                                                         
AAPL    1.000000  0.408329  0.340494  0.426640  0.316591  0.344006  0.664803   
BA      0.408329  1.000000  0.480211  0.469329  0.255429  0.369723  0.385437   
CVX     0.340494  0.480211  1.000000  0.423181  0.336531  0.385836  0.346581   
DIS     0.426640  0.469329  0.423181  1.000000  0.276481  0.371073  0.441457   
JNJ     0.316591  0.255429  0.336531  0.276481  1.000000  0.502654  0.333894   
KO      0.344006  0.369723  0.385836  0.371073  0.502654  1.000000  0.362113   
MSFT    0.664803  0.385437  0.346581  0.441457  0.333894  0.362113  1.000000   
NFLX    0.409397  0.249065  0.156535  0.316824  0.133644  0.138901  0.457790   
TSLA    0.435021  0.316676  0.222118  0.303546  0.095266  0.139512  0.410569   
V       0.559902  0.471014  0.462075  0.532333  0.412348  0.473351  0.612434   

Ticker      NFLX      TSLA         V  


In [62]:
corr_pandas = returns.corr()
print(corr_pandas)

Ticker      AAPL        BA       CVX       DIS       JNJ        KO      MSFT  \
Ticker                                                                         
AAPL    1.000000  0.408329  0.340494  0.426640  0.316591  0.344006  0.664803   
BA      0.408329  1.000000  0.480211  0.469329  0.255429  0.369723  0.385437   
CVX     0.340494  0.480211  1.000000  0.423181  0.336531  0.385836  0.346581   
DIS     0.426640  0.469329  0.423181  1.000000  0.276481  0.371073  0.441457   
JNJ     0.316591  0.255429  0.336531  0.276481  1.000000  0.502654  0.333894   
KO      0.344006  0.369723  0.385836  0.371073  0.502654  1.000000  0.362113   
MSFT    0.664803  0.385437  0.346581  0.441457  0.333894  0.362113  1.000000   
NFLX    0.409397  0.249065  0.156535  0.316824  0.133644  0.138901  0.457790   
TSLA    0.435021  0.316676  0.222118  0.303546  0.095266  0.139512  0.410569   
V       0.559902  0.471014  0.462075  0.532333  0.412348  0.473351  0.612434   

Ticker      NFLX      TSLA         V  


In [63]:
print(np.allclose(corr_manual, corr_pandas))

True


## Verify Mathematical Properties

In [64]:
print(np.allclose(cov_pandas, cov_manual))
print(np.allclose(corr_pandas, corr_manual))

True
True


In [65]:
print(np.linalg.eigvals(cov_pandas))
print(np.linalg.eigvals(corr_pandas))

[2.05662657e-03 7.54190357e-04 5.78077540e-04 3.02886081e-04
 2.11615950e-04 1.76860755e-04 1.28679627e-04 1.03024868e-04
 8.46213641e-05 6.20006124e-05]
[4.40458936 1.26754615 0.86847908 0.31593012 0.3885642  0.66768723
 0.46275342 0.5101591  0.56394509 0.55034625]


In [66]:
print(sum(np.linalg.eigvals(corr_pandas)))

9.999999999999996


In [67]:
print(np.linalg.eigvalsh(cov_pandas).min() >= -1e-10)
print(np.linalg.eigvalsh(corr_pandas).min() >= -1e-10)

True
True


In [68]:
eigenvectors = np.linalg.eigh(corr_pandas)[1]
print(eigenvectors)

[[ 0.51447674 -0.51891858  0.10831348  0.18103599 -0.16813667 -0.38096726
   0.1765733   0.19182773  0.22161398 -0.36198977]
 [-0.08239445  0.08291208  0.40062708  0.65055133 -0.00689208  0.17180911
  -0.06207514 -0.50959614 -0.08397907 -0.32009022]
 [-0.01050373 -0.13744293 -0.30947345 -0.35189631 -0.62454025  0.08292931
  -0.01784042 -0.43936245 -0.28507298 -0.29973696]
 [-0.06421959 -0.24949276  0.17400997 -0.45420232  0.58250253 -0.17861897
  -0.36548194 -0.28330064 -0.02315219 -0.33386184]
 [-0.02745653 -0.00619267  0.54990511 -0.23637886 -0.09541818  0.34196177
   0.14564412  0.46965663 -0.45459687 -0.25777287]
 [-0.01856105 -0.15947263 -0.60757931  0.3208902   0.373739    0.16481756
   0.07387675  0.20861453 -0.44739419 -0.29345495]
 [-0.76085953  0.04152092 -0.03866622  0.0689204  -0.16823707 -0.36425003
   0.03454997  0.26016842  0.21413782 -0.37068259]
 [ 0.09573082 -0.00201377 -0.12988949  0.02951892 -0.13407157  0.52832768
  -0.58257987  0.25072941  0.46051482 -0.24828438]


In [69]:
eigenvectors = np.linalg.eigh(corr_pandas)[1]

eigenvectors_df = pd.DataFrame(
    eigenvectors,
    index=corr_pandas.index,
    columns=["Eigenvector 1", "Eigenvector 2",
             "Eigenvector 3", "Eigenvector 4", "Eigenvector 5", 
            "Eigenvector 6", "Eigenvector 7", "Eigenvector 8",
            "Eigenvector 9", "Eigenvector 10"]
)

print(eigenvectors_df)

        Eigenvector 1  Eigenvector 2  Eigenvector 3  Eigenvector 4  \
Ticker                                                               
AAPL         0.514477      -0.518919       0.108313       0.181036   
BA          -0.082394       0.082912       0.400627       0.650551   
CVX         -0.010504      -0.137443      -0.309473      -0.351896   
DIS         -0.064220      -0.249493       0.174010      -0.454202   
JNJ         -0.027457      -0.006193       0.549905      -0.236379   
KO          -0.018561      -0.159473      -0.607579       0.320890   
MSFT        -0.760860       0.041521      -0.038666       0.068920   
NFLX         0.095731      -0.002014      -0.129889       0.029519   
TSLA        -0.025153       0.066748      -0.072168      -0.208198   
V            0.366726       0.781702      -0.081179      -0.077878   

        Eigenvector 5  Eigenvector 6  Eigenvector 7  Eigenvector 8  \
Ticker                                                               
AAPL        -0.1681

In [70]:
print((corr_pandas >= -1).all().all())
print((corr_pandas <= 1).all().all())

True
True


## Visuals

In [71]:
fig = px.imshow(
    corr_pandas,
    text_auto=".3f",
    color_continuous_scale="piyg_r",
    zmin=-1,
    zmax=1,
    title="10 Asset Correlation Matrix"
)

fig.show()

In [72]:
@interact(stock1=stocks, stock2=stocks, stock3=stocks)
def correlation_heatmap(stock1="AAPL", stock2="TSLA", stock3="MSFT"):

    selected = [stock1, stock2, stock3]

    corr = returns[selected].corr()

    fig = px.imshow(
        corr,
        text_auto=".3f",
        color_continuous_scale="RdPu",
        zmin=-1,
        zmax=1,
        title=f"Correlation Matrix: {stock1}, {stock2}, {stock3}"
    )

    fig.show()

interactive(children=(Dropdown(description='stock1', options=('AAPL', 'TSLA', 'MSFT', 'KO', 'JNJ', 'NFLX', 'V'…

In [73]:
stocks = list(returns.columns)
size = widgets.IntSlider(
    value=5,
    min=2,
    max=min(10, len(stocks)),
    step=1, 
    description="Assets:"
)
selection = widgets.SelectMultiple(
    options=stocks, 
    value=tuple(stocks[:5]),
    description="Stocks",
    rows=10
)

def update_heatmap(n, selected):
    selected = list(selected)
    if len(selected) != n:
        print(f"Please select exactly {n} stocks.")
        return
        
    corr = returns[selected].corr()
    
    fig = px.imshow(
        corr, 
        text_auto=".3f",
        color_continuous_scale="PiYG_r",
        zmin=-1,
        zmax=1,
        title=f"Correlation Matrix ({n} Assets)"
    )

    fig.update_layout(width=700, height=650)
    fig.show()

output = widgets.interactive_output(
    update_heatmap,
    {"n": size, "selected": selection}
)

display(size, selection, output)

IntSlider(value=5, description='Assets:', max=10, min=2)

SelectMultiple(description='Stocks', index=(0, 1, 2, 3, 4), options=('AAPL', 'BA', 'CVX', 'DIS', 'JNJ', 'KO', …

Output()

In [74]:
cov_pandas.to_csv("python_cov_10.csv")
corr_pandas.to_csv("python_corr_10.csv")

# 50 Assets

In [75]:
stocks = [
    "AAPL", "TSLA", "MSFT", "KO", "JNJ", "NFLX", "V", "BA", "DIS", "CVX",
    "NVDA", "AMD", "INTC", "CSCO", "ORCL", "IBM", "CRM", "GOOGL", "META", "CMCSA",
    "AMZN", "HD", "MCD", "NKE", "SBUX","PEP", "PG", "WMT", "COST", "CL","PFE", "MRK", "ABBV", "ABT", "TMO",
    "JPM", "BAC", "WFC", "GS", "MA","XOM", "COP", "SLB","CAT", "DE", "UNP", "UPS","NEE", "DUK","LIN"
]

print(len(stocks))

50


## Logarithmic Daily Returns

In [76]:
data = yf.download(
    stocks, 
    start = "2015-01-01",
    end = "2026-01-01",
    interval = "1d",
    auto_adjust = True # requests prices adjusted for corporote actions like splits and dividends
)

[*********************100%***********************]  50 of 50 completed


In [77]:
print(data.head())
print(data.shape)

Price           Close                                                   \
Ticker           AAPL       ABBV        ABT   AMD     AMZN          BA   
Date                                                                     
2015-01-02  24.171757  40.868271  35.535442  2.67  15.4260  113.657219   
2015-01-05  23.490803  40.099186  35.543365  2.66  15.1095  112.870056   
2015-01-06  23.493008  39.900703  35.139732  2.63  14.7645  111.540627   
2015-01-07  23.822441  41.513363  35.424648  2.58  14.9210  113.272392   
2015-01-08  24.737747  41.947529  36.152763  2.61  15.0230  115.275261   

Price                                                   ...    Volume  \
Ticker            BAC        CAT         CL      CMCSA  ...      SBUX   
Date                                                    ...             
2015-01-02  13.979467  68.735878  52.145664  19.798239  ...   6886000   
2015-01-05  13.573364  65.107544  51.760468  19.394470  ...  11623800   
2015-01-06  13.167256  64.688606  51.21663

In [78]:
prices = data["Close"]

In [79]:
print(prices.head())

Ticker           AAPL       ABBV        ABT   AMD     AMZN          BA  \
Date                                                                     
2015-01-02  24.171757  40.868271  35.535442  2.67  15.4260  113.657219   
2015-01-05  23.490803  40.099186  35.543365  2.66  15.1095  112.870056   
2015-01-06  23.493008  39.900703  35.139732  2.63  14.7645  111.540627   
2015-01-07  23.822441  41.513363  35.424648  2.58  14.9210  113.272392   
2015-01-08  24.737747  41.947529  36.152763  2.61  15.0230  115.275261   

Ticker            BAC        CAT         CL      CMCSA  ...       SBUX  \
Date                                                    ...              
2015-01-02  13.979467  68.735878  52.145664  19.798239  ...  32.076733   
2015-01-05  13.573364  65.107544  51.760468  19.394470  ...  31.462282   
2015-01-06  13.167256  64.688606  51.216637  19.141474  ...  31.206278   
2015-01-07  13.229732  65.691078  51.624516  19.124138  ...  31.974316   
2015-01-08  13.503072  66.364365  52.

In [80]:
print(prices.shape)

(2766, 50)


In [81]:
print(prices.isna().sum())

Ticker
AAPL     0
ABBV     0
ABT      0
AMD      0
AMZN     0
BA       0
BAC      0
CAT      0
CL       0
CMCSA    0
COP      0
COST     0
CRM      0
CSCO     0
CVX      0
DE       0
DIS      0
DUK      0
GOOGL    0
GS       0
HD       0
IBM      0
INTC     0
JNJ      0
JPM      0
KO       0
LIN      0
MA       0
MCD      0
META     0
MRK      0
MSFT     0
NEE      0
NFLX     0
NKE      0
NVDA     0
ORCL     0
PEP      0
PFE      0
PG       0
SBUX     0
SLB      0
TMO      0
TSLA     0
UNP      0
UPS      0
V        0
WFC      0
WMT      0
XOM      0
dtype: int64


In [82]:
returns = np.log(prices / prices.shift(1))
returns = returns.dropna()

In [83]:
print(returns.head())
print(returns.shape)

Ticker          AAPL      ABBV       ABT       AMD      AMZN        BA  \
Date                                                                     
2015-01-05 -0.028576 -0.018998  0.000223 -0.003752 -0.020731 -0.006950   
2015-01-06  0.000094 -0.004962 -0.011421 -0.011342 -0.023098 -0.011848   
2015-01-07  0.013925  0.039621  0.008075 -0.019195  0.010544  0.015407   
2015-01-08  0.037702  0.010404  0.020346  0.011561  0.006813  0.017527   
2015-01-09  0.001072 -0.027736 -0.010563  0.007634 -0.011818 -0.001974   

Ticker           BAC       CAT        CL     CMCSA  ...      SBUX       SLB  \
Date                                                ...                       
2015-01-05 -0.029480 -0.054231 -0.007414 -0.020605  ... -0.019341 -0.027454   
2015-01-06 -0.030376 -0.006455 -0.010562 -0.013131  ... -0.008170 -0.019750   
2015-01-07  0.004734  0.015378  0.007932 -0.000906  ...  0.024314 -0.000123   
2015-01-08  0.020451  0.010197  0.013514  0.021514  ...  0.016008  0.012044   
2015-01

In [84]:
print(returns.describe())

Ticker         AAPL         ABBV          ABT          AMD         AMZN  \
count   2765.000000  2765.000000  2765.000000  2765.000000  2765.000000   
mean       0.000874     0.000612     0.000447     0.001586     0.000979   
std        0.018154     0.016866     0.014807     0.036419     0.020664   
min       -0.137708    -0.177363    -0.102982    -0.277456    -0.151398   
25%       -0.007309    -0.006845    -0.006463    -0.017022    -0.008835   
50%        0.000950     0.001134     0.000701     0.000566     0.001081   
75%        0.009964     0.008816     0.008362     0.020048     0.011362   
max        0.142617     0.128985     0.103783     0.420617     0.132178   

Ticker           BA          BAC          CAT           CL        CMCSA  ...  \
count   2765.000000  2765.000000  2765.000000  2765.000000  2765.000000  ...   
mean       0.000234     0.000490     0.000764     0.000142     0.000107  ...   
std        0.025279     0.019358     0.018898     0.012117     0.016028  ...   
min 

## Sample Covariance Matrix

In [85]:
mean_returns = returns.mean() # calculate mean of each stock
print(mean_returns)
centered_returns = returns - mean_returns
n = len(returns) # no observations
cov_manual = centered_returns.T @ centered_returns / (n-1)

print(cov_manual)

Ticker
AAPL     0.000874
ABBV     0.000612
ABT      0.000447
AMD      0.001586
AMZN     0.000979
BA       0.000234
BAC      0.000490
CAT      0.000764
CL       0.000142
CMCSA    0.000107
COP      0.000239
COST     0.000731
CRM      0.000546
CSCO     0.000491
CVX      0.000280
DE       0.000672
DIS      0.000104
DUK      0.000285
GOOGL    0.000896
GS       0.000624
HD       0.000528
IBM      0.000400
INTC     0.000101
JNJ      0.000358
JPM      0.000698
KO       0.000309
LIN      0.000502
MA       0.000710
MCD      0.000530
META     0.000773
MRK      0.000361
MSFT     0.000903
NEE      0.000502
NFLX     0.001061
NKE      0.000156
NVDA     0.002154
ORCL     0.000595
PEP      0.000271
PFE      0.000104
PG       0.000278
SBUX     0.000343
SLB     -0.000179
TMO      0.000558
TSLA     0.001239
UNP      0.000331
UPS      0.000104
V        0.000631
WFC      0.000304
WMT      0.000567
XOM      0.000264
dtype: float64
Ticker      AAPL      ABBV       ABT       AMD      AMZN        BA       BAC  

In [86]:
cov_pandas = returns.cov()
print(cov_pandas)

Ticker      AAPL      ABBV       ABT       AMD      AMZN        BA       BAC  \
Ticker                                                                         
AAPL    0.000330  0.000085  0.000115  0.000284  0.000209  0.000187  0.000149   
ABBV    0.000085  0.000284  0.000107  0.000097  0.000068  0.000099  0.000103   
ABT     0.000115  0.000107  0.000219  0.000148  0.000100  0.000114  0.000114   
AMD     0.000284  0.000097  0.000148  0.001326  0.000318  0.000261  0.000210   
AMZN    0.000209  0.000068  0.000100  0.000318  0.000427  0.000164  0.000130   
BA      0.000187  0.000099  0.000114  0.000261  0.000164  0.000639  0.000256   
BAC     0.000149  0.000103  0.000114  0.000210  0.000130  0.000256  0.000375   
CAT     0.000135  0.000090  0.000089  0.000210  0.000123  0.000221  0.000220   
CL      0.000065  0.000059  0.000082  0.000058  0.000047  0.000068  0.000062   
CMCSA   0.000116  0.000081  0.000087  0.000147  0.000106  0.000143  0.000142   
COP     0.000134  0.000114  0.000083  0.

In [87]:
print(np.allclose(cov_manual, cov_pandas))

True


## Sample Correlation Matrix

In [88]:
std_returns = returns.std() # sample standard deviation
corr_manual = cov_manual / np.outer(std_returns, std_returns)
print(corr_manual)

Ticker      AAPL      ABBV       ABT       AMD      AMZN        BA       BAC  \
Ticker                                                                         
AAPL    1.000000  0.276890  0.428538  0.429461  0.557659  0.408329  0.423155   
ABBV    0.276890  1.000000  0.430167  0.158326  0.195992  0.233301  0.314425   
ABT     0.428538  0.430167  1.000000  0.274711  0.327763  0.304696  0.396765   
AMD     0.429461  0.158326  0.274711  1.000000  0.422133  0.283606  0.297465   
AMZN    0.557659  0.195992  0.327763  0.422133  1.000000  0.314297  0.325413   
BA      0.408329  0.233301  0.304696  0.283606  0.314297  1.000000  0.522793   
BAC     0.423155  0.314425  0.396765  0.297465  0.325413  0.522793  1.000000   
CAT     0.392994  0.282108  0.317210  0.304728  0.315268  0.461677  0.600707   
CL      0.296736  0.289103  0.454265  0.132215  0.188884  0.223461  0.263769   
CMCSA   0.399871  0.299279  0.366953  0.251644  0.320332  0.353612  0.458960   
COP     0.307105  0.281700  0.233586  0.

In [89]:
corr_pandas = returns.corr()
print(corr_pandas)

Ticker      AAPL      ABBV       ABT       AMD      AMZN        BA       BAC  \
Ticker                                                                         
AAPL    1.000000  0.276890  0.428538  0.429461  0.557659  0.408329  0.423155   
ABBV    0.276890  1.000000  0.430167  0.158326  0.195992  0.233301  0.314425   
ABT     0.428538  0.430167  1.000000  0.274711  0.327763  0.304696  0.396765   
AMD     0.429461  0.158326  0.274711  1.000000  0.422133  0.283606  0.297465   
AMZN    0.557659  0.195992  0.327763  0.422133  1.000000  0.314297  0.325413   
BA      0.408329  0.233301  0.304696  0.283606  0.314297  1.000000  0.522793   
BAC     0.423155  0.314425  0.396765  0.297465  0.325413  0.522793  1.000000   
CAT     0.392994  0.282108  0.317210  0.304728  0.315268  0.461677  0.600707   
CL      0.296736  0.289103  0.454265  0.132215  0.188884  0.223461  0.263769   
CMCSA   0.399871  0.299279  0.366953  0.251644  0.320332  0.353612  0.458960   
COP     0.307105  0.281700  0.233586  0.

In [90]:
print(np.allclose(corr_manual, corr_pandas))

True


## Verify Mathematical Properties

In [91]:
print(np.allclose(cov_pandas, cov_manual))
print(np.allclose(corr_pandas, corr_manual))

True
True


In [92]:
print(np.linalg.eigvals(cov_pandas))
print(np.linalg.eigvals(corr_pandas))

[6.88886623e-03 1.78181410e-03 1.00858906e-03 8.86454971e-04
 6.63548642e-04 4.99579282e-04 4.55303217e-04 4.01755645e-04
 3.81387767e-04 3.48566681e-04 3.00802769e-04 2.90863090e-04
 2.70721516e-04 2.62668699e-04 2.42948015e-04 2.05471598e-04
 1.99230642e-04 1.94718284e-04 1.79534702e-04 1.70444034e-04
 2.52679914e-05 3.18872416e-05 3.41525967e-05 3.61090933e-05
 1.61664893e-04 1.56569477e-04 4.56646548e-05 1.43210462e-04
 1.42474015e-04 4.89325605e-05 5.26943682e-05 5.62541989e-05
 5.84157072e-05 1.33939533e-04 1.28313591e-04 1.27147451e-04
 7.16524620e-05 7.19957898e-05 7.48001536e-05 7.94040529e-05
 8.16860649e-05 8.63975087e-05 9.09655740e-05 9.56330724e-05
 1.00590449e-04 1.20209941e-04 1.16086674e-04 1.14863520e-04
 1.08216854e-04 1.07834282e-04]
[19.26354083  3.59888235  3.37737793  1.43040899  1.27151091  1.08647892
  0.94515966  0.91759987  0.82922948  0.82144067  0.73673323  0.09981063
  0.10589751  0.71122267  0.14260517  0.68278633  0.66630014  0.65089878
  0.16844473  0.6

In [93]:
print(sum(np.linalg.eigvals(corr_pandas)))

50.000000000000036


In [94]:
eigenvectors = np.linalg.eigh(corr_pandas)[1]
print(eigenvectors)

[[-1.94731171e-02  8.00676639e-03 -4.46007699e-02 ... -1.48272472e-01
   1.07681662e-01  1.58760044e-01]
 [ 1.32682047e-03 -1.51200492e-02 -1.68040544e-03 ...  3.69231457e-04
  -1.12216394e-01  1.11229672e-01]
 [-8.68066255e-03  2.41326356e-02 -2.58480634e-02 ... -1.09907831e-01
  -1.20045327e-01  1.45683230e-01]
 ...
 [-4.45329475e-02  1.23253790e-01 -2.76084041e-01 ...  1.99584976e-01
   5.17513961e-02  1.58168864e-01]
 [-5.47313649e-03 -7.39565531e-03 -8.43438982e-03 ... -1.15685957e-01
  -1.25395923e-01  1.05036778e-01]
 [-8.11889156e-02  3.20515127e-02  5.39906989e-01 ...  3.07485142e-01
   5.95712871e-03  1.38609685e-01]]


In [95]:
print(np.linalg.eigvalsh(cov_pandas).min() >= -1e-10)
print(np.linalg.eigvalsh(corr_pandas).min() >= -1e-10)

True
True


In [96]:
eigenvectors = np.linalg.eigh(corr_pandas)[1]

eigenvectors_df = pd.DataFrame(
    eigenvectors,
    index=corr_pandas.index,
    columns=[f"Eigenvector {i}" for i in range(1, 51)]
)

print(eigenvectors_df)

        Eigenvector 1  Eigenvector 2  Eigenvector 3  Eigenvector 4  \
Ticker                                                               
AAPL        -0.019473       0.008007      -0.044601      -0.037238   
ABBV         0.001327      -0.015120      -0.001680      -0.004446   
ABT         -0.008681       0.024133      -0.025848       0.033961   
AMD         -0.014487       0.017946       0.008697       0.003155   
AMZN        -0.009129      -0.003749      -0.036414      -0.034507   
BA          -0.029505      -0.042209       0.026104      -0.019563   
BAC          0.253633      -0.664270       0.240966      -0.362915   
CAT         -0.015986       0.013106      -0.074557      -0.032776   
CL           0.011761      -0.010942       0.029437       0.001792   
CMCSA        0.010513      -0.020167       0.007199      -0.005524   
COP         -0.034776       0.088136       0.136523       0.176105   
COST         0.023756      -0.003586      -0.004640      -0.028536   
CRM         -0.00893

In [97]:
print((corr_pandas >= -1).all().all())
print((corr_pandas <= 1).all().all())

True
True


## Visuals

In [98]:
fig = px.imshow(
    corr_pandas,
    text_auto=".3f",
    color_continuous_scale="piyg_r",
    zmin=-1,
    zmax=1,
    title="50 Asset Correlation Matrix"
)

fig.show()

In [99]:
@interact(stock1=stocks, stock2=stocks, stock3=stocks)
def correlation_heatmap(stock1="AAPL", stock2="TSLA", stock3="MSFT"):

    selected = [stock1, stock2, stock3]

    corr = returns[selected].corr()

    fig = px.imshow(
        corr,
        text_auto=".3f",
        color_continuous_scale="RdPu",
        zmin=-1,
        zmax=1,
        title=f"Correlation Matrix: {stock1}, {stock2}, {stock3}"
    )

    fig.show()

interactive(children=(Dropdown(description='stock1', options=('AAPL', 'TSLA', 'MSFT', 'KO', 'JNJ', 'NFLX', 'V'…

In [100]:
stocks = list(returns.columns)
size = widgets.IntSlider(
    value=5,
    min=2,
    max=min(50, len(stocks)),
    step=1, 
    description="Assets:"
)
selection = widgets.SelectMultiple(
    options=stocks, 
    value=tuple(stocks[:5]),
    description="Stocks",
    rows=10
)

def update_heatmap(n, selected):
    selected = list(selected)
    if len(selected) != n:
        print(f"Please select exactly {n} stocks.")
        return
        
    corr = returns[selected].corr()
    
    fig = px.imshow(
        corr, 
        text_auto=".3f",
        color_continuous_scale="PiYG_r",
        zmin=-1,
        zmax=1,
        title=f"Correlation Matrix ({n} Assets)"
    )

    fig.update_layout(width=700, height=650)
    fig.show()

output = widgets.interactive_output(
    update_heatmap,
    {"n": size, "selected": selection}
)

display(size, selection, output)

IntSlider(value=5, description='Assets:', max=50, min=2)

SelectMultiple(description='Stocks', index=(0, 1, 2, 3, 4), options=('AAPL', 'ABBV', 'ABT', 'AMD', 'AMZN', 'BA…

Output()

In [101]:
cov_pandas.to_csv("python_cov_50.csv")
corr_pandas.to_csv("python_corr_50.csv")